# Phase 5 — Data Preparation for the Dashboard

This notebook bridges the Phase 1-4 outputs (large parquet files) and the Dash dashboard. It runs once, offline, and saves small aggregate files to `notebooks/Phase5/data/`. The dashboard reads only those small files, so it renders instantly without processing 6.3 million rows on the user side.

Run the cells top to bottom using the `DataMining` environment (needs pandas, numpy, scipy, scikit-learn, pyarrow).

## Why pipelining and sampling

The dataset has 6.3 million rows (about 230 MB), which is too large to send to the browser and recompute every time a slider moves. That would be slow and memory-heavy.

We solve this in two layers. First, we pre-aggregate offline in this notebook: we compute the summaries from the full data once and store them as small tables indexed by day (day 1..31), for example transaction and fraud counts per day per cluster. All the files together are only about 0.6 MB. Second, the dashboard simply sums the bins for the selected day range, which is a light operation on small tables, so the latency is close to zero.

The numbers stay correct because ratios (fraud rate, anomaly rate, proportions) are sample-invariant: they do not change with sample size. So ratios computed here from the full data remain representative when the range changes in the dashboard.

We only sample where population precision is not needed: the cluster map (8,000 points), the anomaly detection (1.2 million rows, since Isolation Forest on 6.3 million is too heavy), and the data explorer table (about 5,500 curated rows). Section F below runs a Kolmogorov-Smirnov test to confirm the sample distribution is nearly identical to the population.

In [38]:
# ── Setup & path resolver (robust terhadap cwd) ──────────────────────────────
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

def find_base():
    here = Path.cwd()
    for cand in [here] + list(here.parents):
        if (cand / 'data' / 'processed' / 'real_used' / 'transactions.parquet').exists():
            return cand
    raise FileNotFoundError('Tidak menemukan data/processed/real_used dari cwd: ' + str(here))

BASE = find_base()
DATA = BASE / 'data' / 'processed' / 'real_used'
OUT  = BASE / 'notebooks' / 'Phase5' / 'data'
OUT.mkdir(parents=True, exist_ok=True)
RS = np.random.RandomState(42)   # seed tetap → hasil dapat direproduksi

def W(df, name):
    df.to_csv(OUT / name, index=False)
    print(f'  ->  {name:28s} {len(df):>8,} baris')

print('BASE :', BASE)
print('DATA :', DATA)
print('OUT  :', OUT)

BASE : d:\PPTI\Cawu 5\Data Mining\Lecturer\PhySim_Synthetic_Financial_Fraud
DATA : d:\PPTI\Cawu 5\Data Mining\Lecturer\PhySim_Synthetic_Financial_Fraud\data\processed\real_used
OUT  : d:\PPTI\Cawu 5\Data Mining\Lecturer\PhySim_Synthetic_Financial_Fraud\notebooks\Phase5\data


In [39]:
# ── Muat hasil Phase 1-4 & turunkan kolom bantu ──────────────────────────────
print('Memuat hasil Phase 1-4 ...')
raw  = pd.read_parquet(DATA / 'transactions.parquet')
clus = pd.read_parquet(DATA / 'data_phase2_results.parquet')
lab  = pd.read_parquet(DATA / 'labels_validation.parquet', columns=['isFraud'])

N = len(raw)
y = lab['isFraud'].to_numpy()
raw['cluster'] = clus['cluster_kmeans'].to_numpy()
raw['isFraud'] = y
raw['hour']    = raw['step'] % 24
raw['day']     = ((raw['step'] - 1) // 24 + 1).astype(int)          # 1..31 (PaySim ~30 hari)
raw['drain']   = np.where(raw['oldbalanceOrg'] > 0,
                          (raw['amount'] / raw['oldbalanceOrg']).clip(0, 10), 0)
NDAYS = int(raw['day'].max())
print(f'  {N:,} transaksi | fraud {y.sum():,} ({y.mean()*100:.3f}%) | hari 1..{NDAYS}')

Memuat hasil Phase 1-4 ...
  6,362,620 transaksi | fraud 8,213 (0.129%) | hari 1..31


## Global aggregates (from the full data)

These are the reference summaries used by the KPI cards, the Overview charts, and the cluster profile table. Everything here is computed over all 6.3 million rows, so they are true population figures.

In [40]:
# 1) Distribusi tipe transaksi
g = raw.groupby('type').agg(count=('amount', 'size'), fraud=('isFraud', 'sum'))
g['pct'] = (g['count'] / N * 100).round(2)
g['fraud_rate'] = (g['fraud'] / g['count'] * 100).round(3)
W(g.reset_index(), 'type_distribution.csv')

# 2) Pola per jam (denyut normal harian)
t = raw.groupby('hour').agg(volume=('amount', 'size'), fraud=('isFraud', 'sum'))
t['fraud_rate'] = (t['fraud'] / t['volume'] * 100).round(3)
W(t.reset_index(), 'temporal.csv')

# 3) Agregat per HARI (untuk timeline & slider Overview)
dd = raw.groupby('day').agg(volume=('amount', 'size'), fraud=('isFraud', 'sum'),
                            mean_amount=('amount', 'mean'))
dd['fraud_rate'] = (dd['fraud'] / dd['volume'] * 100).round(3)
dd['mean_amount'] = dd['mean_amount'].round(0)
W(dd.reset_index(), 'daily.csv')

  ->  type_distribution.csv               5 baris
  ->  temporal.csv                       24 baris
  ->  daily.csv                          31 baris


In [41]:
# 4) Profil cluster global + nama bisnis otomatis
cp = raw.groupby('cluster').agg(
    n_records=('amount', 'size'), mean_amount=('amount', 'mean'),
    median_amount=('amount', 'median'), mean_drain=('drain', 'mean'),
    mean_oldbal=('oldbalanceOrg', 'mean'), fraud=('isFraud', 'sum'))
cp = cp.join(raw.groupby('cluster')['type']
             .agg(lambda s: s.value_counts().index[0]).rename('dominant_type'))
cp['pct'] = (cp['n_records'] / N * 100).round(2)
cp['fraud_rate'] = (cp['fraud'] / cp['n_records'] * 100).round(3)
amt_tier = pd.cut(cp['mean_amount'], [-1, 50_000, 300_000, 1e12],
                  labels=['nominal kecil', 'nominal sedang', 'nominal besar'])
cp['name'] = [f"{cp.loc[c, 'dominant_type']} · {amt_tier[c]}" for c in cp.index]
W(cp.reset_index().round(2), 'cluster_profiles.csv')

# 4b) Detail + komposisi tipe top-3 per cluster
comp = raw.groupby('cluster')['type'].apply(
    lambda s: ' · '.join(f'{tt} {v*100:.0f}%'
              for tt, v in s.value_counts(normalize=True).head(3).items()))
cd = cp[['name', 'n_records', 'pct', 'mean_amount', 'median_amount', 'mean_drain',
         'mean_oldbal', 'dominant_type', 'fraud', 'fraud_rate']].copy()
cd['komposisi'] = comp
W(cd.reset_index().round(2), 'cluster_detail.csv')

name_map = {int(c): cp.loc[c, 'name'] for c in cp.index}
cp[['name', 'fraud_rate', 'mean_drain']]

  ->  cluster_profiles.csv                5 baris
  ->  cluster_detail.csv                  5 baris


,name,fraud_rate,mean_drain
cluster,,,
0,CASH_IN · nominal sedang,0.162,0.828254
1,PAYMENT · nominal kecil,0.002,0.703362
2,CASH_OUT · nominal besar,0.002,0.004821
3,CASH_OUT · nominal besar,0.046,7.591480
4,PAYMENT · nominal kecil,0.402,0.289519


## Per-day aggregates

This is the key to the low latency: each dimension is stored per day, so the dashboard only has to sum the bins for the selected range. Still computed from the full data.

In [42]:
# type x day
tbd = raw.groupby(['day', 'type']).agg(n=('amount', 'size'), fraud=('isFraud', 'sum')).reset_index()
W(tbd, 'type_by_day.csv')

# cluster x day (simpan SUM agar rata-rata rentang = sum/n)
cbd = raw.groupby(['day', 'cluster']).agg(
    n=('amount', 'size'), fraud=('isFraud', 'sum'),
    sum_amount=('amount', 'sum'), sum_drain=('drain', 'sum')).reset_index()
cbd['sum_amount'] = cbd['sum_amount'].round(0)
cbd['sum_drain']  = cbd['sum_drain'].round(2)
W(cbd, 'cluster_by_day.csv')

# cluster x type x day (komposisi per rentang)
ctd = raw.groupby(['day', 'cluster', 'type']).size().reset_index(name='n')
W(ctd, 'cluster_type_by_day.csv')

  ->  type_by_day.csv                   152 baris
  ->  cluster_by_day.csv                154 baris
  ->  cluster_type_by_day.csv           654 baris


## Cluster map (2D PCA), 8,000-point sample

We reduce the 13 scaled features down to two axes so the clusters can be plotted. The day column is kept so the points can follow the slider.

In [43]:
print('PCA cluster map (sampel 8.000) ...')
scaled = pd.read_parquet(DATA / 'data_phase2_clustering.parquet')
sidx = RS.choice(N, 8000, replace=False)
p2 = PCA(n_components=2, random_state=42).fit_transform(scaled.values[sidx])
W(pd.DataFrame({'pc1': p2[:, 0].round(3), 'pc2': p2[:, 1].round(3),
                'cluster': raw['cluster'].to_numpy()[sidx],
                'type':    raw['type'].to_numpy()[sidx],
                'day':     raw['day'].to_numpy()[sidx]}), 'cluster_scatter.csv')

PCA cluster map (sampel 8.000) ...
  ->  cluster_scatter.csv             8,000 baris


## Anomaly detection (1.2 million sample) and the explorer table

We rerun the three-method detection from Phase 4 (IQR, Z-score, Isolation Forest) with voting, on a 1.2 million-row sample, because Isolation Forest on 6.3 million is too heavy. For each flagged row we also store the main reason (the feature with the largest absolute z-score) so the dashboard can explain a transaction when it is clicked.

In [44]:
print('Deteksi anomali (sampel 1,2 jt) ...')
aidx = RS.choice(N, 1_200_000, replace=False)
sub = raw.iloc[aidx].copy()
sub['errorBalanceOrig'] = sub['newbalanceOrig'] + sub['amount'] - sub['oldbalanceOrg']
mdest = sub['nameDest'].str.startswith('M')
sub['errorBalanceDest'] = np.where(mdest, 0, sub['oldbalanceDest'] + sub['amount'] - sub['newbalanceDest'])

NUM = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'drain']
FEAT_LABEL = {'amount': 'nominal transaksi sangat ekstrem',
              'oldbalanceOrg': 'saldo awal pengirim sangat ekstrem',
              'oldbalanceDest': 'saldo tujuan sangat ekstrem',
              'errorBalanceOrig': 'saldo pengirim tidak konsisten secara logika',
              'errorBalanceDest': 'saldo tujuan tidak konsisten secara logika',
              'drain': 'menguras hampir seluruh saldo'}

is_iqr = np.zeros(len(sub), int); is_z = np.zeros(len(sub), int); zcols = []
for c in NUM:
    Q1, Q3 = sub[c].quantile(.25), sub[c].quantile(.75); I = Q3 - Q1
    is_iqr += ((sub[c] < Q1 - 3 * I) | (sub[c] > Q3 + 3 * I)).astype(int).values
    zc = np.abs(stats.zscore(sub[c], nan_policy='omit'))
    zcols.append(np.nan_to_num(zc, nan=0.0))
    is_z += (zc > 3).astype(int)
is_iqr = (is_iqr > 0).astype(int); is_z = (is_z > 0).astype(int)
zmat = np.column_stack(zcols)
top_reason = np.array([FEAT_LABEL[NUM[i]] for i in zmat.argmax(axis=1)])

iso = IsolationForest(n_estimators=100, contamination=0.01, max_samples=256, random_state=42, n_jobs=-1)
Xs = scaled.values[aidx]
is_iso = (iso.fit_predict(Xs) == -1).astype(int)
score = -iso.score_samples(Xs)
vote = is_iqr + is_z + is_iso
hc = (vote >= 2).astype(int)
ys = y[aidx]
dist = clus['dist_to_centroid'].to_numpy()[aidx]
print(f'  IQR {is_iqr.mean()*100:.2f}% | Z {is_z.mean()*100:.2f}% | IsoForest {is_iso.mean()*100:.2f}% | high-conf {hc.mean()*100:.2f}%')

Deteksi anomali (sampel 1,2 jt) ...
  IQR 43.06% | Z 4.46% | IsoForest 1.00% | high-conf 5.01%


In [45]:
# Agregat anomali per-HARI (untuk slider di tab Anomaly)
S = pd.DataFrame({'day': raw['day'].to_numpy()[aidx], 'type': raw['type'].to_numpy()[aidx],
                  'cluster': raw['cluster'].to_numpy()[aidx], 'vote': vote, 'hc': hc, 'fraud': ys})
avd = S.groupby(['day', 'vote']).agg(count=('fraud', 'size'), fraud=('fraud', 'sum')).reset_index()
W(avd, 'anomaly_by_day_vote.csv')
atd = S.groupby(['day', 'type']).agg(n=('fraud', 'size'), hc=('hc', 'sum'), fraud=('fraud', 'sum')).reset_index()
W(atd, 'anomaly_by_day_type.csv')
acd = S.groupby(['day', 'cluster']).agg(n=('fraud', 'size'), hc=('hc', 'sum'), fraud=('fraud', 'sum')).reset_index()
W(acd, 'anomaly_by_day_cluster.csv')

  ->  anomaly_by_day_vote.csv           124 baris
  ->  anomaly_by_day_type.csv           152 baris
  ->  anomaly_by_day_cluster.csv        154 baris


In [46]:
# Tabel Jelajah Data (kurasi seimbang: fraud / anomali-nonfraud / normal)
expl = pd.DataFrame({
    'type':         raw['type'].to_numpy()[aidx],
    'segmen':       pd.Series(raw['cluster'].to_numpy()[aidx]).map(name_map).values,
    'day':          raw['day'].to_numpy()[aidx].astype(int),
    'amount':       sub['amount'].round(0).astype('int64').values,
    'saldo_awal':   sub['oldbalanceOrg'].round(0).astype('int64').values,
    'drain_ratio':  sub['drain'].round(3).values,
    'jam':          sub['hour'].values.astype(int),
    'cluster':      raw['cluster'].to_numpy()[aidx],
    'is_iqr': is_iqr, 'is_z': is_z, 'is_iso': is_iso,
    'anomaly_vote': vote, 'dist': np.round(dist, 2),
    'top_reason': top_reason, 'anomali': hc, 'fraud': ys,
})
def _take(d_, k): return d_.sample(min(k, len(d_)), random_state=7)
_expl = pd.concat([
    _take(expl[expl['fraud'] == 1], 1200),
    _take(expl[(expl['anomali'] == 1) & (expl['fraud'] == 0)], 1800),
    _take(expl[(expl['anomali'] == 0) & (expl['fraud'] == 0)], 2500),
]).sample(frac=1, random_state=7).reset_index(drop=True)
W(_expl, 'data_explorer.csv')
print(f'  komposisi: fraud={int(_expl.fraud.sum())}, anomali-nonfraud={int(((_expl.anomali==1)&(_expl.fraud==0)).sum())}, normal={int(((_expl.anomali==0)&(_expl.fraud==0)).sum())}')

  ->  data_explorer.csv               5,500 baris
  komposisi: fraud=1200, anomali-nonfraud=1800, normal=2500


## Rule activity per day

The 12 Apriori rules are a global deliverable; their support, confidence, and lift are computed over all 6.3 million rows. The day slider does not recompute the rules (on very low-volume days the lift becomes unstable) — it only shows how active each pattern is over the selected range.

In [47]:
# salin 12 aturan
rules = pd.read_csv(DATA / 'phase3_association_rules.csv')
W(rules, 'rules.csv')

# hitung berapa kali antecedent, consequent, dan keduanya muncul per hari
rdf = pd.read_parquet(DATA / 'data_phase3_rules.parquet')
rdf['day'] = raw['day'].to_numpy()
day_counts = rdf.groupby('day').size()

def parse_terms(s):
    out = {}
    for part in str(s).split(','):
        part = part.strip()
        if '=' in part:
            k, v = part.split('=', 1)
            out[k.strip()] = v.strip()
    return out

def mask_for(terms):
    m = np.ones(len(rdf), dtype=bool)
    for k, v in terms.items():
        if k in rdf.columns:
            m &= (rdf[k].to_numpy() == v)
    return m

rows = []
for _, r in rules.iterrows():
    A, C = parse_terms(r['antecedent']), parse_terms(r['consequent'])
    mA, mC = mask_for(A), mask_for(C); mAC = mA & mC
    dA  = pd.Series(rdf['day'].to_numpy()[mA]).value_counts()
    dC  = pd.Series(rdf['day'].to_numpy()[mC]).value_counts()
    dAC = pd.Series(rdf['day'].to_numpy()[mAC]).value_counts()
    for d in range(1, NDAYS + 1):
        rows.append({'rule_#': int(r['rule_#']), 'day': d,
                     'n_ac': int(dAC.get(d, 0)), 'n_a': int(dA.get(d, 0)),
                     'n_c': int(dC.get(d, 0)), 'n_total': int(day_counts.get(d, 0))})
W(pd.DataFrame(rows), 'rule_by_day.csv')

  ->  rules.csv                          12 baris
  ->  rule_by_day.csv                   372 baris


## Checking that the sample is representative (Kolmogorov-Smirnov)

This answers the fair question of whether a sample can still be trusted. The KS test compares the sample distribution (1.2 million) with the population (6.3 million) on the key features. The KS statistic is the largest gap between the two cumulative distributions, so a smaller value means the two are more identical. We also compare the fraud rate and the share of each transaction type between the sample and the population.

In [48]:
print('Uji representativeness sampel vs populasi ...')
KSFEAT = ['amount', 'oldbalanceOrg', 'drain']
ks = {}
for c in KSFEAT:
    stat, _ = stats.ks_2samp(sub[c].to_numpy(), raw[c].to_numpy())
    ks[c] = round(float(stat), 4)

type_pop = (raw['type'].value_counts(normalize=True) * 100).round(2)
type_smp = (sub['type'].value_counts(normalize=True) * 100).round(2)
type_cmp = pd.DataFrame({'populasi_%': type_pop, 'sampel_%': type_smp}).fillna(0)
type_cmp['selisih'] = (type_cmp['sampel_%'] - type_cmp['populasi_%']).round(2)

report = {
    'sample_n': int(len(sub)), 'pop_n': int(N),
    'sample_frac_pct': round(len(sub) / N * 100, 2),
    'ks': ks,
    'fraud_rate_pop': round(float(y.mean() * 100), 4),
    'fraud_rate_sample': round(float(ys.mean() * 100), 4),
    'type_pop_pct': {k: float(v) for k, v in type_pop.items()},
    'type_sample_pct': {k: float(v) for k, v in type_smp.items()},
    'verdict': 'Distribusi sampel ≈ populasi (KS sangat kecil) → sampel REPRESENTATIF.',
}
json.dump(report, open(OUT / 'sampling_report.json', 'w'), indent=2)
print('  KS:', ks)
print(f"  fraud rate  populasi={report['fraud_rate_pop']}%  sampel={report['fraud_rate_sample']}%")
type_cmp

Uji representativeness sampel vs populasi ...
  KS: {'amount': 0.0009, 'oldbalanceOrg': 0.0005, 'drain': 0.0006}
  fraud rate  populasi=0.1291%  sampel=0.1312%


,populasi_%,sampel_%,selisih
type,,,
CASH_OUT,35.17,35.19,0.02
PAYMENT,33.81,33.78,-0.03
CASH_IN,21.99,22.00,0.01
TRANSFER,8.38,8.37,-0.01
DEBIT,0.65,0.65,0.00


## Supervision-focus curve (gains curve)

Here we sort the transactions from most suspicious (highest Isolation Forest score) to most normal, then ask: if we only inspect the top X percent, what share of all fraud do we catch. This is what drives the interactive simulator in the Business Insight menu, and it is the clearest way to show that checking a small slice already catches most of the fraud.

In [49]:
order = np.argsort(-score)                       # paling anomali dulu
ys_sorted = ys[order]
cum = np.cumsum(ys_sorted) / max(ys_sorted.sum(), 1) * 100
n = len(ys_sorted)
pcts = np.arange(1, 101)
idx = np.clip((pcts / 100 * n).astype(int) - 1, 0, n - 1)
gains = pd.DataFrame({'pct_inspected': pcts.astype(int), 'pct_fraud_caught': np.round(cum[idx], 2)})
gains = pd.concat([pd.DataFrame({'pct_inspected': [0], 'pct_fraud_caught': [0.0]}), gains], ignore_index=True)
W(gains, 'gains_curve.csv')
print('  tangkapan @10% teratas =', float(gains.loc[gains.pct_inspected == 10, 'pct_fraud_caught'].iloc[0]), '%')

  ->  gains_curve.csv                   101 baris
  tangkapan @10% teratas = 76.37 %


## Summary KPIs and saving kpis.json

In [50]:
auc = roc_auc_score(ys, score)
dec = pd.qcut(pd.Series(score).rank(method='first'), 10, labels=False)
top_recall = ys[dec.values == 9].sum() / ys.sum()
kpis = {
    'total_tx': int(N), 'fraud_count': int(y.sum()), 'fraud_rate': round(float(y.mean() * 100), 3),
    'n_clusters': int(raw['cluster'].nunique()),
    'iqr_pct': round(float(is_iqr.mean() * 100), 2), 'zscore_pct': round(float(is_z.mean() * 100), 2),
    'iso_pct': round(float(is_iso.mean() * 100), 2),
    'high_conf_pct': round(float(hc.mean() * 100), 2), 'high_conf_count_est': int(hc.mean() * N),
    'auc': round(float(auc), 3), 'top_decile_recall': round(float(top_recall * 100), 1),
    'n_rules': int(len(rules)), 'n_days': NDAYS, 'anomaly_sample_n': int(len(sub)),
}
json.dump(kpis, open(OUT / 'kpis.json', 'w'), indent=2)
print(json.dumps(kpis, indent=2))
print('\nSELESAI. Semua berkas agregat tersimpan di:', OUT)
print('Jalankan dashboard:  cd notebooks/Phase5/dashboard  &&  python app.py')

{
  "total_tx": 6362620,
  "fraud_count": 8213,
  "fraud_rate": 0.129,
  "n_clusters": 5,
  "iqr_pct": 43.06,
  "zscore_pct": 4.46,
  "iso_pct": 1.0,
  "high_conf_pct": 5.01,
  "high_conf_count_est": 318857,
  "auc": 0.94,
  "top_decile_recall": 76.4,
  "n_rules": 12,
  "n_days": 31,
  "anomaly_sample_n": 1200000
}

SELESAI. Semua berkas agregat tersimpan di: d:\PPTI\Cawu 5\Data Mining\Lecturer\PhySim_Synthetic_Financial_Fraud\notebooks\Phase5\data
Jalankan dashboard:  cd notebooks/Phase5/dashboard  &&  python app.py


### Output files (written to `notebooks/Phase5/data/`)

`kpis.json` holds the global KPIs. `type_distribution.csv`, `temporal.csv`, and `daily.csv` feed the Overview. The cluster files (profiles, detail, by-day, type-by-day, scatter) feed Segmentation. `rules.csv` and `rule_by_day.csv` feed Association. The anomaly-by-day files and `data_explorer.csv` feed the Anomaly page. `gains_curve.csv` drives the Business Insight simulator, and `sampling_report.json` holds the representativeness check.

Every ratio is computed from the full data or a large sample, so it stays representative when the range changes.